In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer


In [10]:
# ----------------------------
# 1. Load Data
# ----------------------------
df = pd.read_csv("../data/steel_industry_data.csv")

print(df.isna().sum())

date                                    0
Usage_kWh                               0
Lagging_Current_Reactive.Power_kVarh    0
Leading_Current_Reactive_Power_kVarh    0
CO2(tCO2)                               0
Lagging_Current_Power_Factor            0
Leading_Current_Power_Factor            0
NSM                                     0
WeekStatus                              0
Day_of_week                             0
Load_Type                               0
dtype: int64


In [11]:


# ----------------------------
# 2. Date Processing
# ----------------------------
df['date'] = pd.to_datetime(df['date'], format='mixed')
df['hour'] = df['date'].dt.hour
df['month'] = df['date'].dt.month
df.drop('date', axis=1, inplace=True)

# ----------------------------
# 3. Encode Target (Classification)
# ----------------------------
label_encoder = LabelEncoder()
df['Load_Type'] = label_encoder.fit_transform(df['Load_Type'])

# ----------------------------
# 4. Split Features / Target
# ----------------------------
X = df.drop('Load_Type', axis=1)
X = df.drop('WeekStatus', axis=1)
y = df['Load_Type']

# ----------------------------
# 5. Column Groups
# ----------------------------
numerical_cols = [
    'Usage_kWh',
    'Lagging_Current_Reactive.Power_kVarh',
    'Leading_Current_Reactive_Power_kVarh',
    'CO2(tCO2)',
    'Lagging_Current_Power_Factor',
    'Leading_Current_Power_Factor',
    'NSM',
    'hour',
    'month'
]

categorical_cols = ['Day_of_week']


all_cols = numerical_cols + categorical_cols

# ----------------------------
# 6. Preprocessing
# ----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ]
)

# ----------------------------
# 7. Train-Test Split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # important for classification
)

# ----------------------------
# 8. Transform Data
# ----------------------------
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert to DataFrame (optional)
X_train_processed = pd.DataFrame(X_train_processed)
X_test_processed = pd.DataFrame(X_test_processed)

# X_train_processed.columns = all_cols  # Set column names after transformation
# X_test_processed.columns = all_cols

X_train_processed['y'] = y_train.values
X_test_processed['y'] = y_test.values

# ----------------------------
# 9. Save Files
# ----------------------------
X_train_processed.to_csv("../data/steel_preprocessed_train.csv", index=False)
X_test_processed.to_csv("../data/steel_preprocessed_test.csv", index=False)


print("✅ Classification preprocessing complete.")

✅ Classification preprocessing complete.
